In [ ]:
%load_ext autoreload
%autoreload 2

# modal

> Abstractions for creating a Modal Sandbox

In [ ]:
#| default_exp modal

In [ ]:
#| export
import modal

In [ ]:
#| export
def mk_app(name): return modal.App.lookup(name, create_if_missing=True)

In [ ]:
app = mk_app('solveit-sandbox'); app

In [ ]:
#| export
def mk_img(
    apts,   # system packages to apt-install (list or str)
    pips,   # python packages to pip-install (list or str)
): return modal.Image.debian_slim().apt_install(apts).pip_install(pips)

In [ ]:
img = mk_img('openssh-server', 'ipykernel'); img

Image(<function _Image.pip_install.<locals>.build_dockerfile at 0x7df4716e2480>)

In [ ]:
#| export
def mk_sandbox(
    app,        # modal.App to register the Sandbox with
    img,        # modal.Image for the Sandbox environment
    timeout=600, # auto-terminate after this many seconds
):
    return modal.Sandbox.create('sleep', 'infinity', app=app, image=img, timeout=timeout, unencrypted_ports=[22])

In [ ]:
sb = mk_sandbox(app, img); sb

Sandbox()

In [ ]:
#| export
def get_tunnel(sb):
    t = sb.tunnels()[22]
    return t.unencrypted_host, t.unencrypted_port

In [ ]:
host,port = get_tunnel(sb); host,port

('r446.modal.host', 37801)

In [ ]:
#| export
import os

In [ ]:
#| export
def get_pubkey(): return os.environ['SSH_PUBKEY']

In [ ]:
#| export
def inject_pubkey(sb, pubkey):
    sb.exec('mkdir', '-p', '/root/.ssh')
    sb.exec('bash', '-c', f'echo {pubkey} > /root/.ssh/authorized_keys')

In [ ]:
inject_pubkey(sb, get_pubkey())

In [ ]:
#| export
from time import sleep

In [ ]:
#| export
def start_ssh(sb):
    sb.exec('service', 'ssh', 'start')
    for i in range(20):
        proc = sb.exec('service', 'ssh', 'status')
        if 'fail' in proc.stdout.read(): sleep(0.3)
        elif 'run' in proc.stdout.read(): 
            print('✔ started ssh service')
            return
    print('✖ failed to start ssh service')

In [ ]:
start_ssh(sb)

✔ started ssh service


In [ ]:
#| export
from fastcore.all import *

In [ ]:
#| export
def mk_ssh(host, port):
    return lambda *cmd: run('ssh', '-p', str(port),
        '-o', 'StrictHostKeyChecking=no',
        '-o', 'UserKnownHostsFile=/dev/null',
        '-o', 'ConnectTimeout=10',
        f'root@{host}', *cmd)

In [ ]:
ssh = _ssh(host,port); ssh

<function __main__._ssh.<locals>.<lambda>(*cmd)>

In [ ]:
#| export
def verify_connection(ssh):
    "Verify SSH connection to Sandbox and print system info."
    h, u, w = ssh('hostname; uname -srmo; whoami').splitlines()[:3]
    sys_name, ver, arch, os_name = u.split()
    print(f'系统：  {sys_name}')
    print(f'主机名：{h}')
    print(f'用户：  {w}')
    print(f'内核：  {ver}')
    print(f'架构：  {arch}')
    print(f'系统类型：{os_name}')


In [ ]:
verify_connection(ssh)

系统：  Linux
主机名：modal
用户：  root
内核：  4.4.0
架构：  x86_64
系统类型：GNU/Linux


In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()